WELCOME TO MY PROJECT:

Lightweight Reproduction and Analysis of EcoMapper

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
DATA_DIR = Path(
    r"C:\Users\HP\Downloads\ecomapper-data-lite-v2"
)

print("Dataset exists:", DATA_DIR.exists())

print("\nDataset contents:")

for item in DATA_DIR.iterdir():
    print(" -", item.name)

In [ ]:
for split in ["train", "val", "test"]:

    split_dir = DATA_DIR / split

    json_files = list(split_dir.rglob("*.json"))
    png_files = list(split_dir.rglob("*.png"))

    print(f"\n{split.upper()}")
    print("JSON files:", len(json_files))
    print("PNG files :", len(png_files))

In [ ]:
json_files = list(
    (DATA_DIR / "test").rglob("*.json")
)

print("Number of test JSON files:", len(json_files))

sample_json = json_files[0]

print("\nSample JSON:")
print(sample_json)

In [ ]:
with open(
    sample_json,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)

print(
    json.dumps(
        metadata,
        indent=4
    )
)

In [ ]:
sample_image = sample_json.with_suffix(".png")

print("JSON:")
print(sample_json)

print("\nImage:")
print(sample_image)

print("\nImage exists:")
print(sample_image.exists())

In [ ]:
img = Image.open(sample_image)

print("Image size:", img.size)
print("Image mode:", img.mode)

plt.figure(figsize=(6, 6))

plt.imshow(img)

plt.axis("off")

plt.title(sample_image.stem)

plt.show()

In [ ]:
print("TOP-LEVEL FIELDS:")

for key in metadata.keys():
    print(" -", key)

In [ ]:
print("CLIMATE VARIABLES:")

for key in metadata["climate_data"].keys():
    print(" -", key)

In [ ]:
for variable, values in metadata["climate_data"].items():

    print("\n" + variable)

    print("Type:", type(values))

    print("Value:", values)

In [ ]:
print("\nClimate variable details:\n")

for variable, values in metadata["climate_data"].items():

    print(variable + ":")

    print("Type:", type(values))

    if isinstance(values, dict):

        print(
            "Number of observations:",
            len(values)
        )

        print(
            "First 5 values:",
            list(values.items())[:5]
        )

    else:

        print(
            "Value:",
            values
        )

    print()

In [ ]:
print("Center coordinates:")

print(
    metadata["center_coordinates"]
)

In [ ]:
climate = metadata["climate_data"]

summary = {
    "temperature_mean": np.mean(
        list(climate["T2M"].values())
    ),

    "uv_mean": np.mean(
        list(
            climate["ALLSKY_SFC_UV_INDEX"].values()
        )
    ),

    "solar_radiation_mean": np.mean(
        list(
            climate["ALLSKY_SFC_SW_DWN"].values()
        )
    ),

    "cloud_amount_mean": np.mean(
        list(
            climate["CLOUD_AMT"].values()
        )
    ),

    "SPI": climate["SPI"]
}

print("Monthly climate summary:")

for key, value in summary.items():

    print(
        f"{key}: {value:.3f}"
    )

In [ ]:
def extract_metadata(json_path, split):

    try:

        with open(
            json_path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)


        climate = data.get(
            "climate_data",
            {}
        )

        def monthly_mean(variable):

            values = climate.get(
                variable,
                {}
            )

            # Expected format:
            # {"20170101": value, ...}

            if isinstance(values, dict):

                valid_values = []

                for value in values.values():

                    if value is None:
                        continue

                    try:
                        value = float(value)

                    except (TypeError, ValueError):
                        continue

                    # Dataset missing-value sentinel
                    if value == -999:
                        continue

                    valid_values.append(value)


                if len(valid_values) > 0:

                    return float(
                        np.mean(valid_values)
                    )


            return np.nan


        # ------------------------------------------
        # Coordinates
        # ------------------------------------------

        coordinates = data.get(
            "center_coordinates",
            [np.nan, np.nan]
        )


        if (
            isinstance(coordinates, list)
            and len(coordinates) >= 2
        ):

            coordinate_1 = coordinates[0]
            coordinate_2 = coordinates[1]

        else:

            coordinate_1 = np.nan
            coordinate_2 = np.nan


        # ------------------------------------------
        # Image path
        # ------------------------------------------

        image_path = json_path.with_suffix(
            ".png"
        )


        spi = climate.get(
            "SPI",
            np.nan
        )


        try:

            if spi is not None:

                spi = float(spi)

                if spi == -999:

                    spi = np.nan

        except (TypeError, ValueError):

            spi = np.nan



        return {

            "image_id":
                json_path.stem,

            "split":
                split,

            "image_path":
                str(image_path),

            "json_path":
                str(json_path),

            "land_cover":
                data.get("type"),

            "country":
                data.get("country_code"),

            "date":
                data.get("date"),

            "coordinate_1":
                coordinate_1,

            "coordinate_2":
                coordinate_2,

            "temperature":
                monthly_mean("T2M"),

            "uv_index":
                monthly_mean(
                    "ALLSKY_SFC_UV_INDEX"
                ),

            "solar_radiation":
                monthly_mean(
                    "ALLSKY_SFC_SW_DWN"
                ),

            "cloud_amount":
                monthly_mean(
                    "CLOUD_AMT"
                ),

            "SPI":
                spi,

            "cloud_coverage":
                data.get(
                    "cloud_coverage"
                ),

            "caption":
                data.get(
                    "caption"
                )
        }


    except Exception as e:

        print(
            f"ERROR reading {json_path}: {e}"
        )

        return None

In [ ]:
records = []

for split in ["train", "val", "test"]:

    split_dir = DATA_DIR / split

    json_files = list(
        split_dir.rglob("*.json")
    )

    print(
        f"{split}: {len(json_files)} JSON files"
    )


    for json_path in json_files:

        record = extract_metadata(
            json_path,
            split
        )

        if record is not None:

            records.append(record)


metadata_df = pd.DataFrame(
    records
)


print("\nTotal samples:")

print(
    len(metadata_df)
)


print("\nSamples by split:")

print(
    metadata_df["split"].value_counts()
)

In [ ]:
print(
    metadata_df.head()
)
print(
    metadata_df.columns.tolist()
)

In [ ]:
print("SPI availability by split:")

print(
    metadata_df
    .groupby("split")["SPI"]
    .apply(
        lambda x:
        x.notna().value_counts()
    )
)

In [ ]:
print(
    "\nSPI availability by land cover:"
)

print(
    metadata_df
    .groupby("land_cover")["SPI"]
    .apply(
        lambda x:
        x.notna().value_counts()
    )
)

In [ ]:
OUTPUT_FILE = (
    DATA_DIR /
    "ecomapper_metadata_clean.csv"
)

metadata_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved metadata table to:"
)

print(
    OUTPUT_FILE
)

In [ ]:
print("SPI availability by split:\n")

spi_by_split = (
    metadata_df
    .groupby("split")["SPI"]
    .agg(
        total="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

spi_by_split["available_percent"] = (
    spi_by_split["available"]
    / spi_by_split["total"]
    * 100
)

spi_by_split["missing_percent"] = (
    spi_by_split["missing"]
    / spi_by_split["total"]
    * 100
)

print(spi_by_split)

In [ ]:
print("SPI availability by land cover:\n")

spi_by_landcover = (
    metadata_df
    .groupby("land_cover")["SPI"]
    .agg(
        total="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

spi_by_landcover["available_percent"] = (
    spi_by_landcover["available"]
    / spi_by_landcover["total"]
    * 100
)

print(spi_by_landcover)

In [ ]:
print("Countries with the most missing SPI:\n")

spi_by_country = (
    metadata_df
    .groupby("country")["SPI"]
    .agg(
        total="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

spi_by_country["missing_percent"] = (
    spi_by_country["missing"]
    / spi_by_country["total"]
    * 100
)

print(
    spi_by_country
    .sort_values(
        "missing_percent",
        ascending=False
    )
    .head(30)
)

In [ ]:
spi_by_date = (
    metadata_df
    .groupby("date")["SPI"]
    .agg(
        total="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

spi_by_date["missing_percent"] = (
    spi_by_date["missing"]
    / spi_by_date["total"]
    * 100
)

print(spi_by_date.head(20))

In [ ]:
spi_by_location = (
    metadata_df
    .groupby(
        ["coordinate_1", "coordinate_2"]
    )["SPI"]
    .agg(
        total="size",
        available="count",
        missing=lambda x: x.isna().sum()
    )
)

spi_by_location["available_percent"] = (
    spi_by_location["available"]
    / spi_by_location["total"]
    * 100
)

print(
    "Number of locations:",
    len(spi_by_location)
)

print("\nLocations with SPI available 0%:")
print(
    spi_by_location[
        spi_by_location["available_percent"] == 0
    ].head(20)
)

print("\nLocations with SPI available 100%:")
print(
    spi_by_location[
        spi_by_location["available_percent"] == 100
    ].head(20)
)

DATASET CHECKUP DONE LETS GET THE MODEL TRAINEDDD

In [ ]:
%pip install torch torchvision torchaudio

In [ ]:
import sys
import os

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nCurrent working directory:")
print(os.getcwd())

In [ ]:
%pip install diffusers transformers accelerate safetensors sentencepiece peft

In [ ]:
import torch
import diffusers
import transformers
import accelerate
import peft

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)

In [ ]:
import pandas as pd

def create_ecomapper_prompt(row):
    """
    Create a simplified EcoMapper-style climate-conditioned prompt.

    Note:
    The paper uses temperature, precipitation, and solar radiation.
    Our Lite dataset does not currently provide the same precipitation
    variable, so precipitation is intentionally not fabricated.
    """

    land_cover = str(row["land_cover"])
    country = str(row["country"])
    date = str(row["date"])

    temperature = row["temperature"]
    solar = row["solar_radiation"]

    prompt = (
        f"A satellite image of {land_cover} landscape in {country} "
        f"on {date}. "
        f"The average temperature was {temperature:.2f} degrees Celsius, "
        f"with an average daily solar radiation of {solar:.2f}."
    )

    return prompt

In [ ]:
sample = metadata_df.iloc[0]

print("IMAGE ID:")
print(sample["image_id"])

print("\nPROMPT:")
print(create_ecomapper_prompt(sample))

In [ ]:
sample = metadata_df[metadata_df["image_id"] == "01_2017-01"].iloc[0]

print("Country:", sample["country"])
print("Land cover:", sample["land_cover"])
print("Date:", sample["date"])
print("Temperature:", sample["temperature"])
print("UV:", sample["uv_index"])
print("Solar radiation:", sample["solar_radiation"])
print("Cloud amount:", sample["cloud_amount"])
print("SPI:", sample["SPI"])
print("Coordinates:", sample["coordinate_1"], sample["coordinate_2"])

In [ ]:
import json

json_path = sample["json_path"]

with open(json_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("Climate variables in original JSON:")
for key, value in raw["climate_data"].items():
    print("\n", key)
    print(value)

In [ ]:
import diffusers
import transformers
import accelerate
import peft

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

print(whoami()["name"])

In [ ]:
from huggingface_hub import scan_cache_dir

cache_info = scan_cache_dir()

for repo in cache_info.repos:
    if "stable-diffusion-v1-5" in str(repo.repo_id):
        print(repo.repo_id)
        print(repo.repo_path)

BUILD FINAL MODEL DATASET

In [ ]:


import pandas as pd
import numpy as np

# Make a copy so we don't accidentally modify the original dataframe
model_df = metadata_df.copy()


model_df["latitude"] = model_df["coordinate_1"]
model_df["longitude"] = model_df["coordinate_2"]


numeric_columns = [
    "temperature",
    "uv_index",
    "solar_radiation",
    "cloud_amount",
    "SPI",
    "latitude",
    "longitude"
]

for col in numeric_columns:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce"
    )


def create_model_prompt(row):

    prompt = (
        f"A satellite image of {row['land_cover']} landscape "
        f"in {row['country']} during {row['date']}. "
        f"Average temperature: {row['temperature']:.2f} degrees Celsius. "
        f"Average UV index: {row['uv_index']:.2f}. "
        f"Average solar radiation: {row['solar_radiation']:.2f}. "
        f"Average cloud amount: {row['cloud_amount']:.2f} percent."
    )

    return prompt


model_df["prompt"] = model_df.apply(
    create_model_prompt,
    axis=1
)



model_columns = [
    "image_id",
    "image_path",
    "json_path",
    "split",
    "land_cover",
    "country",
    "date",
    "latitude",
    "longitude",
    "temperature",
    "uv_index",
    "solar_radiation",
    "cloud_amount",
    "SPI",
    "prompt"
]

model_df = model_df[model_columns]


print("=" * 60)
print("FINAL MODEL DATASET")
print("=" * 60)

print("Number of samples:", len(model_df))

print("\nSamples by split:")
print(model_df["split"].value_counts())

print("\nLand-cover distribution:")
print(model_df["land_cover"].value_counts())

print("\nMissing values:")
print(model_df.isna().sum())

print("\nExample prompt:")
print(model_df.iloc[0]["prompt"])

In [ ]:


output_path = "ecomapper_model_metadata.csv"

model_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(model_df))

In [ ]:
import os

model_df["image_exists"] = model_df["image_path"].apply(
    os.path.exists
)

print("Images found:",
      model_df["image_exists"].sum())

print("Images missing:",
      (~model_df["image_exists"]).sum())

CREATE MODEL SPLITS

In [ ]:

train_df = model_df[
    model_df["split"] == "train"
].copy()

val_df = model_df[
    model_df["split"] == "val"
].copy()

test_df = model_df[
    model_df["split"] == "test"
].copy()

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

print("\nUnique locations:")
print("Train:", train_df[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Val:", val_df[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Test:", test_df[["latitude", "longitude"]].drop_duplicates().shape[0])

In [ ]:


from sklearn.preprocessing import StandardScaler

climate_features = [
    "temperature",
    "uv_index",
    "solar_radiation",
    "cloud_amount"
]

scaler = StandardScaler()

# Fit ONLY on training data
train_df[climate_features] = scaler.fit_transform(
    train_df[climate_features]
)

# Apply same transformation to validation/test
val_df[climate_features] = scaler.transform(
    val_df[climate_features]
)

test_df[climate_features] = scaler.transform(
    test_df[climate_features]
)

print("Climate features normalized.")

print("\nTraining statistics:")
print(train_df[climate_features].describe())

In [ ]:
import numpy as np

def get_conditioning_vector(row):
    return np.array([
        row["temperature"],
        row["uv_index"],
        row["solar_radiation"],
        row["cloud_amount"]
    ], dtype=np.float32)


example_vector = get_conditioning_vector(
    train_df.iloc[0]
)

print("Example conditioning vector:")
print(example_vector)

print("Shape:", example_vector.shape)

In [ ]:
land_cover_classes = sorted(
    train_df["land_cover"].unique()
)

land_cover_to_id = {
    land_cover: i
    for i, land_cover in enumerate(land_cover_classes)
}

print("Land-cover mapping:")

for name, idx in land_cover_to_id.items():
    print(idx, "=", name)


train_df["land_cover_id"] = train_df["land_cover"].map(
    land_cover_to_id
)

val_df["land_cover_id"] = val_df["land_cover"].map(
    land_cover_to_id
)

test_df["land_cover_id"] = test_df["land_cover"].map(
    land_cover_to_id
)

In [ ]:
print(
    train_df[
        [
            "image_id",
            "land_cover",
            "land_cover_id",
            "temperature",
            "uv_index",
            "solar_radiation",
            "cloud_amount",
            "prompt"
        ]
    ].head(5).to_string(index=False)
)

In [ ]:

from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np
import os


class EcoMapperDataset(Dataset):

    def __init__(self, dataframe, image_size=256):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        
        image_path = row["image_path"]

        image = Image.open(image_path).convert("RGB")

        # Resize
        image = image.resize(
            (self.image_size, self.image_size)
        )

        # Convert to numpy
        image = np.array(image).astype(np.float32)

        # Normalize image from [0,255] to [-1,1]
        image = image / 127.5 - 1.0

        # HWC → CHW
        image = np.transpose(
            image,
            (2, 0, 1)
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        )

       

        climate = torch.tensor(
            [
                row["temperature"],
                row["uv_index"],
                row["solar_radiation"],
                row["cloud_amount"]
            ],
            dtype=torch.float32
        )

       

        land_cover = torch.tensor(
            row["land_cover_id"],
            dtype=torch.long
        )

      

        return {
            "image": image,
            "climate": climate,
            "land_cover": land_cover,
            "prompt": row["prompt"],
            "image_id": row["image_id"]
        }

In [ ]:

train_dataset = EcoMapperDataset(
    train_df
)

val_dataset = EcoMapperDataset(
    val_df
)

test_dataset = EcoMapperDataset(
    test_df
)


train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)


print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
batch = next(iter(train_loader))

print("IMAGE")
print("Shape:", batch["image"].shape)
print("dtype:", batch["image"].dtype)

print("\nCLIMATE")
print("Shape:", batch["climate"].shape)
print("Values:")
print(batch["climate"])

print("\nLAND COVER")
print("Shape:", batch["land_cover"].shape)
print("Values:")
print(batch["land_cover"])

print("\nPROMPTS")
for prompt in batch["prompt"]:
    print("-", prompt)

print("\nIMAGE IDS")
print(batch["image_id"])

In [ ]:
import matplotlib.pyplot as plt

images = batch["image"]

plt.figure(figsize=(12, 3))

for i in range(4):

    image = images[i].numpy()

    # CHW → HWC
    image = np.transpose(
        image,
        (1, 2, 0)
    )

    # [-1,1] → [0,1]
    image = (image + 1) / 2

    plt.subplot(1, 4, i + 1)
    plt.imshow(image)
    plt.axis("off")
    plt.title(batch["land_cover"][i].item())

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

T = 100

betas = torch.linspace(
    1e-4,
    0.02,
    T,
    device=device
)

alphas = 1.0 - betas

alpha_bars = torch.cumprod(
    alphas,
    dim=0
)

print("Timesteps:", T)
print("First beta:", betas[0].item())
print("Last beta:", betas[-1].item())

In [ ]:
def add_noise(images, timesteps):

    noise = torch.randn_like(images)

    alpha_bar = alpha_bars[timesteps]

    alpha_bar = alpha_bar.view(
        -1, 1, 1, 1
    )

    noisy_images = (
        torch.sqrt(alpha_bar) * images
        +
        torch.sqrt(1 - alpha_bar) * noise
    )

    return noisy_images, noise

In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)

batch_size = images.shape[0]

timesteps = torch.randint(
    0,
    T,
    (batch_size,),
    device=device
)

noisy_images, noise = add_noise(
    images,
    timesteps
)

print("Original image shape:", images.shape)
print("Noisy image shape:", noisy_images.shape)
print("Noise shape:", noise.shape)
print("Timesteps:", timesteps)

In [ ]:
plt.figure(figsize=(12, 3))

for i in range(4):

    image = noisy_images[i].detach().cpu().numpy()

    image = np.transpose(
        image,
        (1, 2, 0)
    )

    image = (image + 1) / 2
    image = np.clip(image, 0, 1)

    plt.subplot(1, 4, i + 1)
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"t={timesteps[i].item()}")

plt.tight_layout()
plt.show()

In [ ]:
class TinyConditionalDenoiser(nn.Module):

    def __init__(
        self,
        num_land_covers=3,
        climate_dim=4
    ):
        super().__init__()

        # Climate embedding
        self.climate_embedding = nn.Sequential(
            nn.Linear(climate_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        # Land-cover embedding
        self.land_cover_embedding = nn.Embedding(
            num_land_covers,
            32
        )

        # Timestep embedding
        self.time_embedding = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        # Combine conditioning
        self.condition_projection = nn.Linear(
            96,
            32
        )

        # Image network
        self.conv1 = nn.Conv2d(
            3,
            32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            32,
            64,
            kernel_size=3,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            64,
            32,
            kernel_size=3,
            padding=1
        )

        self.output = nn.Conv2d(
            32,
            3,
            kernel_size=3,
            padding=1
        )

    def forward(
        self,
        x,
        timestep,
        climate,
        land_cover
    ):

        # Condition embeddings
        climate_emb = self.climate_embedding(
            climate
        )

        land_emb = self.land_cover_embedding(
            land_cover
        )

        time_emb = self.time_embedding(
            timestep.float().unsqueeze(1) / T
        )

        condition = torch.cat(
            [
                climate_emb,
                land_emb,
                time_emb
            ],
            dim=1
        )

        condition = self.condition_projection(
            condition
        )

        # Turn condition into feature maps
        condition = condition.unsqueeze(
            -1
        ).unsqueeze(
            -1
        )

        condition = condition.expand(
            -1,
            -1,
            x.shape[-2],
            x.shape[-1]
        )

        # Image processing
        h = F.relu(
            self.conv1(x)
        )

        h = torch.cat(
            [h, condition],
            dim=1
        )

        # Reduce back to 64 channels
        h = h[:, :64]

        h = F.relu(
            self.conv2(h)
        )

        h = F.relu(
            self.conv3(h)
        )

        return self.output(h)

In [ ]:
model = TinyConditionalDenoiser(
    num_land_covers=len(land_cover_to_id),
    climate_dim=4
).to(device)

print(model)
num_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Trainable parameters:",
    f"{num_parameters:,}"
)


In [ ]:
class TinyConditionalDenoiser(nn.Module):

    def __init__(
        self,
        num_land_covers=3,
        climate_dim=4
    ):
        super().__init__()

    
        self.climate_embedding = nn.Sequential(
            nn.Linear(climate_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        self.land_cover_embedding = nn.Embedding(
            num_land_covers,
            32
        )

       
        self.time_embedding = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        
        self.condition_projection = nn.Linear(
            96,
            32
        )

        

        # RGB → 32
        self.conv1 = nn.Conv2d(
            3,
            32,
            kernel_size=3,
            padding=1
        )

        # 32 image channels + 32 condition channels
        # = 64 channels
        self.conv2 = nn.Conv2d(
            64,
            64,
            kernel_size=3,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            64,
            32,
            kernel_size=3,
            padding=1
        )

        # 32 → RGB noise prediction
        self.output = nn.Conv2d(
            32,
            3,
            kernel_size=3,
            padding=1
        )

    def forward(
        self,
        x,
        timestep,
        climate,
        land_cover
    ):

        # -------------------------------
        # Climate embedding
        # -------------------------------

        climate_emb = self.climate_embedding(
            climate
        )

        # -------------------------------
        # Land-cover embedding
        # -------------------------------

        land_emb = self.land_cover_embedding(
            land_cover
        )

        # -------------------------------
        # Timestep embedding
        # -------------------------------

        time_emb = self.time_embedding(
            timestep.float().unsqueeze(1) / T
        )

        # -------------------------------
        # Combine all conditions
        # -------------------------------

        condition = torch.cat(
            [
                climate_emb,
                land_emb,
                time_emb
            ],
            dim=1
        )

        condition = self.condition_projection(
            condition
        )

        # [batch, 32]
        # →
        # [batch, 32, 256, 256]

        condition = condition.unsqueeze(-1).unsqueeze(-1)

        condition = condition.expand(
            -1,
            -1,
            x.shape[-2],
            x.shape[-1]
        )

        # -------------------------------
        # Image features
        # -------------------------------

        h = F.relu(
            self.conv1(x)
        )

        # h = [B, 32, H, W]
        # condition = [B, 32, H, W]
        #
        # concatenate → [B, 64, H, W]

        h = torch.cat(
            [h, condition],
            dim=1
        )

        h = F.relu(
            self.conv2(h)
        )

        h = F.relu(
            self.conv3(h)
        )

        # Predict noise
        output = self.output(h)

        return output

In [ ]:
model = TinyConditionalDenoiser(
    num_land_covers=len(land_cover_to_id),
    climate_dim=4
).to(device)

num_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("Model created successfully.")
print("Trainable parameters:", f"{num_parameters:,}")

In [ ]:
climate = batch["climate"].to(device)

land_cover = batch["land_cover"].to(device)

predicted_noise = model(
    noisy_images,
    timesteps,
    climate,
    land_cover
)

print("Predicted noise shape:")
print(predicted_noise.shape)

In [ ]:
loss = F.mse_loss(
    predicted_noise,
    noise
)

print("Initial diffusion loss:", loss.item())

In [ ]:
# ==========================================
# STEP 54.1 — OPTIMIZER
# ==========================================

import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

print("Optimizer created.")

In [ ]:
# ==========================================
# STEP 54.2 — TINY DIFFUSION TRAINING
# ==========================================

num_steps = 100

loss_history = []

model.train()

train_iterator = iter(train_loader)

for step in range(num_steps):

    # --------------------------------------
    # Get batch
    # --------------------------------------

    try:
        batch = next(train_iterator)

    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    # --------------------------------------
    # Move data to device
    # --------------------------------------

    images = batch["image"].to(device)
    climate = batch["climate"].to(device)
    land_cover = batch["land_cover"].to(device)

    # --------------------------------------
    # Random diffusion timestep
    # --------------------------------------

    timesteps = torch.randint(
        0,
        T,
        (images.shape[0],),
        device=device
    )

    # --------------------------------------
    # Add noise
    # --------------------------------------

    noisy_images, noise = add_noise(
        images,
        timesteps
    )

    # --------------------------------------
    # Predict noise
    # --------------------------------------

    predicted_noise = model(
        noisy_images,
        timesteps,
        climate,
        land_cover
    )

    # --------------------------------------
    # Diffusion loss
    # --------------------------------------

    loss = F.mse_loss(
        predicted_noise,
        noise
    )

    # --------------------------------------
    # Backpropagation
    # --------------------------------------

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    # --------------------------------------
    # Store loss
    # --------------------------------------

    loss_history.append(
        loss.item()
    )

    # --------------------------------------
    # Print progress
    # --------------------------------------

    if (step + 1) % 10 == 0:

        recent_loss = np.mean(
            loss_history[-10:]
        )

        print(
            f"Step {step + 1:03d}/{num_steps} "
            f"| Loss: {loss.item():.4f} "
            f"| Avg: {recent_loss:.4f}"
        )

In [ ]:
# ==========================================
# STEP 54.3 — LOSS CURVE
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    loss_history
)

plt.xlabel("Training step")
plt.ylabel("MSE noise prediction loss")
plt.title("Tiny Conditional Diffusion Training")

plt.grid(True)

plt.show()

In [ ]:
# ==========================================
# STEP 54.4 — SAVE CHECKPOINT
# ==========================================

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss_history": loss_history,
    "land_cover_to_id": land_cover_to_id,
    "climate_features": climate_features,
    "T": T
}

torch.save(
    checkpoint,
    "ecomapper_tiny_diffusion.pth"
)

print("Checkpoint saved.")

In [ ]:
# ==========================================
# STEP 55.1 — CLIMATE CLASSIFICATION DATA
# ==========================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

climate_features = [
    "temperature",
    "uv_index",
    "solar_radiation",
    "cloud_amount"
]

target_column = "land_cover_id"

X_train = train_df[climate_features].values
y_train = train_df[target_column].values

X_val = val_df[climate_features].values
y_val = val_df[target_column].values

X_test = test_df[climate_features].values
y_test = test_df[target_column].values

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
# ==========================================
# STEP 55.2 — STANDARDIZATION
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Training mean:")
print(X_train_scaled.mean(axis=0))

print("\nTraining standard deviation:")
print(X_train_scaled.std(axis=0))

In [ ]:
# ==========================================
# STEP 55.3 — CLIMATE CLASSIFIER
# ==========================================

class ClimateClassifier(nn.Module):

    def __init__(self, input_dim=4, num_classes=3):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 32),

            nn.ReLU(),

            nn.Linear(32, 32),

            nn.ReLU(),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):

        return self.network(x)

In [ ]:
classifier = ClimateClassifier(
    input_dim=4,
    num_classes=3
).to(device)

print(classifier)

In [ ]:
# ==========================================
# STEP 55.4 — PYTORCH TENSORS
# ==========================================

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
).to(device)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
).to(device)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
).to(device)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
).to(device)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
).to(device)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
).to(device)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

In [ ]:
# ==========================================
# STEP 55.5 — TRAIN CLASSIFIER
# ==========================================

classifier_optimizer = optim.AdamW(
    classifier.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()

classifier_losses = []

epochs = 100

for epoch in range(epochs):

    classifier.train()

    logits = classifier(
        X_train_tensor
    )

    loss = criterion(
        logits,
        y_train_tensor
    )

    classifier_optimizer.zero_grad()

    loss.backward()

    classifier_optimizer.step()

    classifier_losses.append(
        loss.item()
    )

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1:03d}/{epochs} "
            f"| Loss: {loss.item():.4f}"
        )

In [ ]:
# ==========================================
# STEP 55.6 — EVALUATION
# ==========================================

classifier.eval()

with torch.no_grad():

    train_logits = classifier(
        X_train_tensor
    )

    val_logits = classifier(
        X_val_tensor
    )

    test_logits = classifier(
        X_test_tensor
    )

train_predictions = train_logits.argmax(
    dim=1
).cpu().numpy()

val_predictions = val_logits.argmax(
    dim=1
).cpu().numpy()

test_predictions = test_logits.argmax(
    dim=1
).cpu().numpy()


train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("Train accuracy:", train_accuracy)
print("Validation accuracy:", val_accuracy)
print("Test accuracy:", test_accuracy)

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "agricultural",
            "urban",
            "wild"
        ]
    )
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    classifier_losses
)

plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Climate-only Land-cover Classifier")

plt.grid(True)

plt.show()

In [ ]:
# ==========================================
# STEP 56.1 — CHECK IMAGE BATCH
# ==========================================

batch = next(iter(train_loader))

print("Batch keys:")
print(batch.keys())

print("\nImage shape:")
print(batch["image"].shape)

print("\nLand-cover shape:")
print(batch["land_cover"].shape)

print("\nLand-cover values:")
print(batch["land_cover"])

In [ ]:
# ==========================================
# STEP 56.2 — IMAGE-ONLY CNN
# ==========================================

class ImageClassifier(nn.Module):

    def __init__(self, num_classes=3):

        super().__init__()

        self.features = nn.Sequential(

            # 256 -> 128
            nn.Conv2d(
                3, 16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 128 -> 64
            nn.Conv2d(
                16, 32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 64 -> 32
            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 32 -> 16
            nn.Conv2d(
                64, 128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 16 * 16,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [ ]:
image_classifier = ImageClassifier(
    num_classes=3
).to(device)

num_parameters = sum(
    p.numel()
    for p in image_classifier.parameters()
)

print(
    "Trainable parameters:",
    f"{num_parameters:,}"
)

In [ ]:
# ==========================================
# STEP 56.4 — FORWARD PASS TEST
# ==========================================

images = batch["image"].to(device)

logits = image_classifier(images)

print("Input:", images.shape)
print("Output:", logits.shape)

In [ ]:
# ==========================================
# STEP 56.5 — TRAIN IMAGE CLASSIFIER
# ==========================================

image_optimizer = optim.AdamW(
    image_classifier.parameters(),
    lr=1e-3
)

image_criterion = nn.CrossEntropyLoss()

image_train_losses = []

epochs = 5

for epoch in range(epochs):

    image_classifier.train()

    running_loss = 0.0
    num_batches = 0

    for batch in train_loader:

        images = batch["image"].to(device)
        labels = batch["land_cover"].to(device)

        logits = image_classifier(images)

        loss = image_criterion(
            logits,
            labels
        )

        image_optimizer.zero_grad()

        loss.backward()

        image_optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    epoch_loss = (
        running_loss / num_batches
    )

    image_train_losses.append(
        epoch_loss
    )

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Loss: {epoch_loss:.4f}"
    )

In [ ]:
# ==========================================
# STEP 56.6 — EVALUATE IMAGE MODEL
# ==========================================

def evaluate_image_classifier(
    model,
    loader,
    device
):

    model.eval()

    predictions = []
    labels_all = []

    with torch.no_grad():

        for batch in loader:

            images = batch["image"].to(device)
            labels = batch["land_cover"].to(device)

            logits = model(images)

            predictions.extend(
                logits.argmax(dim=1)
                .cpu()
                .numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

    predictions = np.array(predictions)
    labels_all = np.array(labels_all)

    accuracy = accuracy_score(
        labels_all,
        predictions
    )

    return accuracy, labels_all, predictions

In [ ]:
val_accuracy, val_labels, val_predictions = (
    evaluate_image_classifier(
        image_classifier,
        val_loader,
        device
    )
)

test_accuracy, test_labels, test_predictions = (
    evaluate_image_classifier(
        image_classifier,
        test_loader,
        device
    )
)

print("Validation accuracy:", val_accuracy)
print("Test accuracy:", test_accuracy)

In [ ]:
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=[
            "agricultural",
            "urban",
            "wild"
        ]
    )
)

In [ ]:
# ==========================================
# STEP 56.8 — BASELINE COMPARISON
# ==========================================

print("=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

print(
    f"Climate-only test accuracy: "
    f"{test_accuracy:.4f}"
)

print(
    "Image-only test accuracy: "
    f"{test_accuracy:.4f}"
)

In [ ]:
climate_test_accuracy = test_accuracy
image_test_accuracy = test_accuracy

print("=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

print(
    f"Climate-only: {climate_test_accuracy:.4f}"
)

print(
    f"Image-only:   {image_test_accuracy:.4f}"
)

In [ ]:
# ==========================================
# VERIFY CLIMATE-ONLY RESULT
# ==========================================

classifier.eval()

with torch.no_grad():

    climate_logits = classifier(
        X_test_tensor
    )

climate_predictions = (
    climate_logits
    .argmax(dim=1)
    .cpu()
    .numpy()
)

climate_test_accuracy = accuracy_score(
    y_test,
    climate_predictions
)

print(
    "Climate-only test accuracy:",
    climate_test_accuracy
)

In [ ]:
# ==========================================
# VERIFY IMAGE-ONLY RESULT
# ==========================================

image_test_accuracy, image_test_labels, image_test_predictions = (
    evaluate_image_classifier(
        image_classifier,
        test_loader,
        device
    )
)

print(
    "Image-only test accuracy:",
    image_test_accuracy
)

In [ ]:
print("\n" + "=" * 50)
print("FINAL BASELINE RESULTS")
print("=" * 50)

print(
    f"Climate-only : {climate_test_accuracy:.4f}"
)

print(
    f"Image-only   : {image_test_accuracy:.4f}"
)

In [ ]:
print("IMAGE-ONLY CLASSIFICATION REPORT")
print("=" * 50)

print(
    classification_report(
        image_test_labels,
        image_test_predictions,
        target_names=[
            "agricultural",
            "urban",
            "wild"
        ]
    )
)

In [ ]:
# ==========================================
# FINAL IMAGE-ONLY ACCURACY
# ==========================================

image_test_accuracy, image_test_labels, image_test_predictions = (
    evaluate_image_classifier(
        image_classifier,
        test_loader,
        device
    )
)

print(
    "Image-only test accuracy:",
    image_test_accuracy
)

In [ ]:
print("\nFINAL BASELINES")
print("=" * 50)
print(f"Climate-only : {climate_test_accuracy:.4f}")
print(f"Image-only   : {image_test_accuracy:.4f}")

In [ ]:
from sklearn.metrics import classification_report

print("=" * 60)
print("CLIMATE-ONLY")
print("=" * 60)

print(
    classification_report(
        y_test,
        climate_predictions,
        target_names=[
            "agricultural",
            "urban",
            "wild"
        ]
    )
)

print("=" * 60)
print("IMAGE-ONLY")
print("=" * 60)

print(
    classification_report(
        image_test_labels,
        image_test_predictions,
        target_names=[
            "agricultural",
            "urban",
            "wild"
        ]
    )
)

In [ ]:
# ==========================================
# STEP 60.1 — IMAGE + CLIMATE MODEL
# ==========================================

class ImageClimateClassifier(nn.Module):

    def __init__(self, num_classes=3):

        super().__init__()

        # ------------------------------
        # IMAGE BRANCH
        # ------------------------------

        self.image_features = nn.Sequential(

            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(
                128 * 16 * 16,
                128
            ),

            nn.ReLU()
        )

        # ------------------------------
        # CLIMATE BRANCH
        # ------------------------------

        self.climate_features = nn.Sequential(

            nn.Linear(4, 32),

            nn.ReLU(),

            nn.Linear(32, 32),

            nn.ReLU()
        )

        # ------------------------------
        # FUSION
        # ------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                128 + 32,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(
        self,
        image,
        climate
    ):

        image_features = self.image_features(
            image
        )

        climate_features = self.climate_features(
            climate
        )

        combined = torch.cat(
            [
                image_features,
                climate_features
            ],
            dim=1
        )

        output = self.classifier(
            combined
        )

        return output

In [ ]:
multimodal_model = ImageClimateClassifier(
    num_classes=3
).to(device)

print(multimodal_model)

In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)

climate = batch["climate"].to(device)

labels = batch["land_cover"].to(device)

logits = multimodal_model(
    images,
    climate
)

print("Images:", images.shape)
print("Climate:", climate.shape)
print("Output:", logits.shape)

In [ ]:
# ==========================================
# STEP 61 — TRAIN IMAGE + CLIMATE MODEL
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim


# Make sure model is on the correct device
multimodal_model = multimodal_model.to(device)


# Loss function
criterion = nn.CrossEntropyLoss()


# Optimizer
optimizer = optim.Adam(
    multimodal_model.parameters(),
    lr=1e-3
)


num_epochs = 5


for epoch in range(num_epochs):

    # ------------------------------
    # TRAINING
    # ------------------------------

    multimodal_model.train()

    train_correct = 0
    train_total = 0
    train_loss = 0.0

    for batch in train_loader:

        images = batch["image"].to(device)
        climate = batch["climate"].to(device)
        labels = batch["land_cover"].to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = multimodal_model(
            images,
            climate
        )

        # Loss
        loss = criterion(
            outputs,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        train_loss += loss.item()

        predictions = outputs.argmax(
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_accuracy = (
        train_correct / train_total
    )


    # ------------------------------
    # VALIDATION
    # ------------------------------

    multimodal_model.eval()

    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():

        for batch in val_loader:

            images = batch["image"].to(device)
            climate = batch["climate"].to(device)
            labels = batch["land_cover"].to(device)

            outputs = multimodal_model(
                images,
                climate
            )

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            predictions = outputs.argmax(
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    val_accuracy = (
        val_correct / val_total
    )


    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss / len(train_loader):.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss / len(val_loader):.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

In [ ]:
print("Climate columns:")
print(metadata_df[[
    "temperature",
    "uv_index",
    "solar_radiation",
    "cloud_amount"
]].describe())

print("\nMissing values:")
print(metadata_df[[
    "temperature",
    "uv_index",
    "solar_radiation",
    "cloud_amount"
]].isna().sum())